In [ ]:
from flask import Flask, request, jsonify
project_API = Flask(__name__)

In [ ]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
mongo_db = client["Data_analysis_project"]
sales_collection = mongo_db["Sales_Collection"]

In [ ]:
@project_API.route('/top_customer')
def top_customer():
    pipline = [
        {
        "$project":{"_id": 0,"CustomerName": 1,"TotalSpent": 1}
        },
    {"$sort": {"TotalSpent": -1}},
    {"$limit": 5}
    ]
    data = list(sales_collection.aggregate(pipline))
    return jsonify(data)

In [ ]:
@project_API.route('/best_selling_product')
def best_selling_product():
    pipeline = [
        {"$unwind": "$TopProducts"},
        {"$group": {
            "_id": {
                "Product_Name": "$TopProducts.Product_Name",
                "Price": "$TopProducts.Price"
            },
            "Quantity": {"$sum": "$TopProducts.Quantity"}
        }},
        {"$sort": {"Quantity": -1}},
        {"$project": {
            "_id": 0,
            "Product_Name": "$_id.Product_Name",
            "Price": "$_id.Price",
            "Quantity": 1
        }},
    {"$limit": 15}
    ]

    data = list(sales_collection.aggregate(pipeline))
    return jsonify(data)


In [ ]:
@project_API.route('/best_product_by_branch')
def best_product_by_branch():
    pipline = [
        {"$unwind":"$TopProducts"},
        {"$group":{
            "_id": {"Product_Name":"$TopProducts.Product_Name",
                    "Branch":"$PreferredBranch"},
            "Quantity": {"$sum":"$TopProducts.Quantity"}
        }},
        {"$project":{
            "_id":0,
            "Product_Name":"$_id.Product_Name",
            "Branch":"$_id.Branch",
            "Quantity":1
        }},
    {"$limit": 15}
    ]
    data = list(sales_collection.aggregate(pipline))
    return jsonify(data)

In [ ]:
@project_API.route("/revenue-by-branch")
def revenue_by_branch():
    pipeline = [
        {"$group": {
            "_id": "$PreferredBranch",
            "Revenue": {"$sum": "$TotalSpent"}
        }},
        {"$project": {
            "_id": 0,
            "PreferredBranch": "$_id",
            "Revenue": 1
        }},
        {"$sort": {"Revenue": -1}}
    ]
    data = list(sales_collection.aggregate(pipeline))
    return jsonify(data)

In [ ]:
@project_API.route("/monthly-sales")
def monthly_sales():
    pipeline = [
        {"$project": {
            "MonthlyPurchases": {"$objectToArray": "$MonthlyPurchases"}
        }},
        {"$unwind": "$MonthlyPurchases"},
        {"$group": {
            "_id": "$MonthlyPurchases.k",
            "TotalPurchases": {"$sum": "$MonthlyPurchases.v"}
        }},
        {"$project": {
            "_id": 0,
            "Month": "$_id",
            "TotalPurchases": 1
        }},
        {"$sort": {"Month": 1}}
    ]
    data = list(sales_collection.aggregate(pipeline))
    return jsonify(data)

In [ ]:
@project_API.route("/seasonal-demand")
def seasonal_demand():
    pipeline = [
        {"$unwind":"$TopProducts"},
        {"$project":{
            "Product_Name":"$TopProducts.Product_Name",
            "MonthlyPurchases":{"$objectToArray":"$MonthlyPurchases"}}
        },
        {"$unwind":"$MonthlyPurchases"},

        {"$group":{
            "_id": {
                "Month":"$MonthlyPurchases.k",
                "Product_Name":"$Product_Name"
            },
            "Sales": {"$sum": "$MonthlyPurchases.v"}}},

        {"$project": {
                "_id": 0,
                "Product": "$_id.Product_Name",
                "Month": "$_id.Month",
                "Sales": 1
                }}
    ]
    data = list(sales_collection.aggregate(pipeline))
    return jsonify(data)

In [ ]:
@project_API.route("/stock-planning")
def stock_planning():
    pipeline = [
        {"$group": {
            "_id": "$PreferredBranch",
            "SalesVolume": {"$sum": "$TotalOrders"},
            "Revenue": {"$sum": "$TotalSpent"}
        }},
        {"$project": {
            "_id": 0,
            "Branch": "$_id",        # بدل " $_id.$PreferredBranch "
            "SalesVolume": 1,
            "Revenue": 1
        }}
    ]
    data = list(sales_collection.aggregate(pipeline))
    return jsonify(data)

# call API

In [ ]:
import threading
import time

def run_flask():
    project_API.run(port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_flask).start()
time.sleep(1)

In [ ]:
import requests
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
base_url = "http://localhost:5000"   # عدّل لو البورت مختلف

def get_df(endpoint: str) -> pd.DataFrame:
    r = requests.get(base_url + endpoint)
    r.raise_for_status()
    return pd.DataFrame(r.json())

In [ ]:
df_top_cust = get_df("/top_customer")


df_top_cust = df_top_cust.sort_values("TotalSpent", ascending=True)

fig = px.bar(
    df_top_cust,
    x="TotalSpent",
    y="CustomerName",
    orientation="h",
    title="Top Customers (Highest Spenders)",
    labels={"TotalSpent": "TotalSpent", "CustomerName": "Customers"}
)

fig.show()

In [ ]:
df_best_prod = get_df("/best_selling_product")
df_best_prod = df_best_prod.sort_values("Quantity", ascending=False)

fig = px.bar(
    df_best_prod,
    x="Product_Name",
    y="Quantity",
    title="Best-Selling Products (Overall)",
    labels={"Product_Name": "Product", "Quantity": "Quantity Sold"},
    hover_data=["Price"]  # يظهر السعر في الـ hover
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
df_prod_branch = get_df("/best_product_by_branch")

# رسم مباشر Grouped Bar: المنتج على الـ x، الفرع لون، y = Quantity
fig = px.bar(
    df_prod_branch,
    x="Product_Name",
    y="Quantity",
    color="Branch",
    barmode="group",
    title="Best-Selling Products by Branch",
    labels={"Product_Name": "Product", "Quantity": "Quantity Sold", "Branch": "Branch"}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
df_rev_branch = get_df("/revenue-by-branch")

fig = px.bar(
    df_rev_branch,
    x="PreferredBranch",
    y="Revenue",
    title="Branch Revenue Comparison",
    labels={"PreferredBranch": "Branch", "Revenue": "Revenue"}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
df_monthly = get_df("/monthly-sales")

# Month string -> datetime عشان الترتيب والمحور الزمني
df_monthly["Month"] = pd.to_datetime(df_monthly["Month"])
df_monthly = df_monthly.sort_values("Month")

fig = px.line(
    df_monthly,
    x="Month",
    y="TotalPurchases",
    title="Monthly Sales Trend",
    labels={"Month": "Month", "TotalPurchases": "Total Purchases"},
    markers=True
)
fig.show()

In [ ]:
df_seasonal = get_df("/seasonal-demand")
df_seasonal["Month"] = pd.to_datetime(df_seasonal["Month"])
df_seasonal = df_seasonal.sort_values(["Product", "Month"])

In [ ]:
df_seasonal["MonthStr"] = df_seasonal["Month"].dt.to_period("M").astype(str)

pivot_seasonal = df_seasonal.pivot_table(
    index="Product",
    columns="MonthStr",
    values="Sales",
    aggfunc="sum"
).fillna(0)

fig = px.imshow(
    pivot_seasonal,
    labels=dict(x="Month", y="Product", color="Sales"),
    x=pivot_seasonal.columns,
    y=pivot_seasonal.index,
    title="Seasonal Product Demand – Heatmap"
)
fig.show()

In [ ]:
00df_stock = get_df("/stock-planning")

fig = px.scatter(
    df_stock,
    x="SalesVolume",
    y="Revenue",
    color="Branch",
    size="SalesVolume",  # حجم النقطة حسب حجم المبيعات
    hover_name="Branch",
    title="Stock Planning – Branch Performance",
    labels={"SalesVolume": "Sales Volume (Orders)", "Revenue": "Revenue"}
)
fig.show()